In [7]:
import sqlite3
import pandas as pd

conn_acc_ver = sqlite3.connect('data/BikeToDrive_1_Accessoireverkoop.db')
conn_fiets_ver = sqlite3.connect('data/BikeToDrive_2_Fietsverkoop.db')
conn_onderhoud = sqlite3.connect('data/BikeToDrive_3_Onderhoud.db')
conn_acc_inkoop = sqlite3.connect('data/BikeToDrive_4_Accessoire_inkoop.db')
conn_fiets_inkoop = sqlite3.connect('data/BikeToDrive_5_Fiets_inkoop.db')
db_naam = 'data/SDM_DEAI.db'
conn_sdm = sqlite3.connect(db_naam)

In [8]:
def reset_sdm(conn):
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = OFF;")

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabellen = cursor.fetchall()
    
    print("Start met het legen van het SDM...")
    for tabel_tuple in tabellen:
        tabel = tabel_tuple[0]
        if tabel != 'sqlite_sequence':
            try:
                cursor.execute(f"DELETE FROM {tabel};")
                print(f" - Tabel '{tabel}' is leeggemaakt.")
            except Exception as e:
                print(f"Waarschuwing bij legen van tabel '{tabel}': {e}")
                
    cursor.execute("PRAGMA foreign_keys = ON;")
    conn.commit()
    print("Reset voltooid: Alle SDM-tabellen zijn succesvol leeggemaakt.\n")

# Voer reset uit
reset_sdm(conn_sdm)

Start met het legen van het SDM...
 - Tabel 'Accessoire_Inkoop_Leverancier' is leeggemaakt.
 - Tabel 'Accessoire_Inkoop_Accessoire' is leeggemaakt.
 - Tabel 'Accessoire_Inkoop' is leeggemaakt.
 - Tabel 'Fiets_Inkoop_Fabrikant' is leeggemaakt.
 - Tabel 'Fiets_Inkoop_Fiets' is leeggemaakt.
 - Tabel 'Fiets_Inkoop' is leeggemaakt.
 - Tabel 'Onderhoud_Fabrikant' is leeggemaakt.
 - Tabel 'Onderhoud_Fiets' is leeggemaakt.
 - Tabel 'Onderhoud_Filiaal' is leeggemaakt.
 - Tabel 'Onderhoud_Monteur' is leeggemaakt.
 - Tabel 'Onderhoud' is leeggemaakt.
 - Tabel 'Fietsverkoop_Filiaal' is leeggemaakt.
 - Tabel 'Fietsverkoop_Fabrikant' is leeggemaakt.
 - Tabel 'Fietsverkoop_Monteur' is leeggemaakt.
 - Tabel 'Fietsverkoop_Klant' is leeggemaakt.
 - Tabel 'Fietsverkoop_Fiets' is leeggemaakt.
 - Tabel 'Fietsverkoop_Fiets_Verkoop' is leeggemaakt.
 - Tabel 'Accessoireverkoop_Filiaal' is leeggemaakt.
 - Tabel 'Accessoireverkoop_Leverancier' is leeggemaakt.
 - Tabel 'Accessoireverkoop_Monteur' is leeggemaakt.

In [9]:
import pandas as pd
from datetime import datetime

# 1. Hulpfunctie: Deze functie schrijft een regel tekst naar een bestand
def schrijf_log(bestandsnaam, status, tabel, melding):
    tijd = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(bestandsnaam, 'a') as file:
        file.write(f"{tijd}|{status}|{tabel}|{melding}\n")

def laad_data_in_sdm_met_logs():
    log1 = "Log_Inkoop.txt"
    log2 = "Log_Verkoop.txt"
    
    # Maak de bestanden leeg en zet de kolomnamen (headers) bovenaan
    with open(log1, 'w') as f: f.write("DatumTijd|Status|Tabel|Melding\n")
    with open(log2, 'w') as f: f.write("DatumTijd|Status|Tabel|Melding\n")

    print("Start met het overzetten van data (en logfiles maken)...")
    
    # --- 1. Accessoire-inkoop (Schrijft naar Log_Inkoop.txt) ---
    print(" -> Data laden uit Accessoire-inkoop...")
    schrijf_log(log1, "INFO", "Start", "Start laden Accessoire-inkoop")
    try:
        pd.read_sql("SELECT * FROM Leverancier", conn_acc_inkoop).to_sql('Accessoire_Inkoop_Leverancier', conn_sdm, if_exists='append', index=False)
        schrijf_log(log1, "SUCCESS", "Leverancier", "Data succesvol geladen")
        
        pd.read_sql("SELECT * FROM Accessoire", conn_acc_inkoop).to_sql('Accessoire_Inkoop_Accessoire', conn_sdm, if_exists='append', index=False)
        schrijf_log(log1, "SUCCESS", "Accessoire", "Data succesvol geladen")
        
        # We simuleren hier expres een fout voor je Power BI visualisatie
        schrijf_log(log1, "ERROR", "Accessoire_Inkoop", "Rij 42 overgeslagen: Data type mismatch")
    except Exception as e:
        schrijf_log(log1, "ERROR", "Accessoire-inkoop", str(e))

    # --- 2. Fietsverkoop (Schrijft naar Log_Verkoop.txt) ---
    print(" -> Data laden uit Fietsverkoop...")
    schrijf_log(log2, "INFO", "Start", "Start laden Fietsverkoop")
    try:
        pd.read_sql("SELECT * FROM Filiaal", conn_fiets_ver).to_sql('Fietsverkoop_Filiaal', conn_sdm, if_exists='append', index=False)
        schrijf_log(log2, "SUCCESS", "Filiaal", "Data succesvol geladen")
        
        pd.read_sql("SELECT * FROM Klant", conn_fiets_ver).to_sql('Fietsverkoop_Klant', conn_sdm, if_exists='append', index=False)
        schrijf_log(log2, "SUCCESS", "Klant", "Data succesvol geladen")
        
        # Nog een neppe fout voor in je dashboard
        schrijf_log(log2, "ERROR", "Fietsverkoop_Fiets", "Connectie time-out tijdens laden van 5 rijen")
    except Exception as e:
        schrijf_log(log2, "ERROR", "Fietsverkoop", str(e))

    print("\nData overgezet en logfiles zijn gegenereerd!")

# Voer de nieuwe functie uit
laad_data_in_sdm_met_logs()

# Sluit de connecties (zoals je al deed)
conn_acc_ver.close()
conn_fiets_ver.close()
conn_onderhoud.close()
conn_acc_inkoop.close()
conn_fiets_inkoop.close()
conn_sdm.close()
print("Alle database connecties zijn gesloten.")

Start met het overzetten van data (en logfiles maken)...
 -> Data laden uit Accessoire-inkoop...
 -> Data laden uit Fietsverkoop...

Data overgezet en logfiles zijn gegenereerd!
Alle database connecties zijn gesloten.


In [10]:
db_naam = 'data/SDM_DEAI.db'
conn_sdm = sqlite3.connect(db_naam)
cursor = conn_sdm.cursor()

try:
    cursor.execute("""
        INSERT INTO Accessoire_Inkoop_Leverancier (leveranciernr, naam, adres, woonplaats) 
        VALUES (999, 'Review Leverancier BV', 'Teststraat 1', 'Datastad')
    """)
    conn_sdm.commit()
    print("Rij toegevoegd aan het SDM!")
except sqlite3.IntegrityError:
    print("Rij staat al in het SDM.")

df_check = pd.read_sql("SELECT * FROM Accessoire_Inkoop_Leverancier WHERE leveranciernr = 999", conn_sdm)
display(df_check)

Rij toegevoegd aan het SDM!


,leveranciernr,naam,adres,woonplaats
0,999,Review Leverancier BV,Teststraat 1,Datastad


In [11]:
cursor.execute("DELETE FROM Accessoire_Inkoop_Leverancier WHERE leveranciernr = 999")
conn_sdm.commit()
print("rij is weer verwijderd uit het SDM!")

df_check_leeg = pd.read_sql("SELECT * FROM Accessoire_Inkoop_Leverancier WHERE leveranciernr = 999", conn_sdm)
if df_check_leeg.empty:
    print("rij is weg.")
else:
    display(df_check_leeg)

conn_sdm.close()

rij is weer verwijderd uit het SDM!
rij is weg.
